# Avaliação Baseline vs Conteúdo

Este notebook compara dois métodos de recomendação:
- Baseline (popularidade/média por item)
- Conteúdo (categorias dos negócios)

Métricas usadas: `RMSE`, `Precision@K` e `Recall@K`.

In [1]:
import json
import math
import os
import random
import time
from collections import Counter, defaultdict

import pandas as pd

## Carregamento de dados

In [2]:
def load_line_json(path):
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                yield json.loads(line)


def load_reviews(path):
    user_ratings = defaultdict(dict)
    for record in load_line_json(path):
        user_id = record.get("user_id")
        item_id = record.get("business_id")
        rating = record.get("stars")
        if user_id is None or item_id is None or rating is None:
            continue
        user_ratings[user_id][item_id] = float(rating)
    return user_ratings


def load_business_categories(path):
    categories = {}
    for record in load_line_json(path):
        item_id = record.get("business_id")
        raw_categories = record.get("categories") or ""
        categories[item_id] = {
            part.strip().lower()
            for part in raw_categories.split(",")
            if part.strip()
        }
    return categories

## Preparação de treino e estatísticas

In [3]:
def split_train_test(user_ratings, min_ratings=5, test_ratio=0.2, seed=42):
    random.seed(seed)
    train = {}
    test = []

    for user_id, ratings in user_ratings.items():
        if len(ratings) < min_ratings:
            train[user_id] = ratings.copy()
            continue

        items = list(ratings.items())
        n_test = max(1, int(len(items) * test_ratio))
        test_items = random.sample(items, n_test)
        test_map = {item: rating for item, rating in test_items}

        train[user_id] = {item: rating for item, rating in items if item not in test_map}
        for item, rating in test_items:
            test.append((user_id, item, rating))

    return train, test


def build_item_stats(train_user_ratings):
    item_counts = Counter()
    item_sum = Counter()

    for ratings in train_user_ratings.values():
        for item_id, rating in ratings.items():
            item_counts[item_id] += 1
            item_sum[item_id] += rating

    item_mean = {
        item_id: item_sum[item_id] / item_counts[item_id]
        for item_id in item_counts
    }
    global_mean = sum(item_sum.values()) / sum(item_counts.values()) if item_counts else 0.0

    return item_counts, item_mean, global_mean

In [4]:
def jaccard_similarity(a, b):
    if not a or not b:
        return 0.0
    inter = len(a & b)
    if inter == 0:
        return 0.0
    union = len(a | b)
    return inter / union


def build_user_profile_categories(user_id, train_user_ratings, categories):
    profile = set()
    for item_id in train_user_ratings.get(user_id, {}):
        profile |= categories.get(item_id, set())
    return profile


def predict_baseline_rating(item_id, item_mean, global_mean):
    return item_mean.get(item_id, global_mean)


def predict_content_rating(user_id, item_id, train_user_ratings, categories, global_mean):
    user_items = train_user_ratings.get(user_id, {})
    if not user_items:
        return global_mean

    target_cats = categories.get(item_id, set())
    weighted_sum = 0.0
    weight_total = 0.0

    for rated_item, rating in user_items.items():
        sim = jaccard_similarity(target_cats, categories.get(rated_item, set()))
        if sim > 0:
            weighted_sum += sim * rating
            weight_total += sim

    if weight_total > 0:
        return weighted_sum / weight_total

    user_mean = sum(user_items.values()) / len(user_items)
    return user_mean

## Ranking e métricas (RMSE, Precision@K, Recall@K)

In [5]:
def rank_baseline(user_id, train_user_ratings, candidates, item_mean, top_k):
    rated = set(train_user_ratings.get(user_id, {}))
    ranked = [item for item in candidates if item not in rated]
    return ranked[:top_k]


def rank_content(user_id, train_user_ratings, categories, all_items, top_k):
    rated = set(train_user_ratings.get(user_id, {}))
    profile = build_user_profile_categories(user_id, train_user_ratings, categories)

    scores = []
    for item_id in all_items:
        if item_id in rated:
            continue
        score = jaccard_similarity(profile, categories.get(item_id, set()))
        scores.append((score, item_id))

    scores.sort(key=lambda x: x[0], reverse=True)
    return [item for _, item in scores[:top_k]]

In [6]:
def evaluate_rmse(test_set, predictor, label="RMSE", progress_step=10):
    mse = 0.0
    count = 0
    total = len(test_set)

    if total == 0:
        print(f"{label}: 100% (no test data)")
        return 0.0

    next_progress = progress_step
    started_at = time.perf_counter()

    for index, (user_id, item_id, actual) in enumerate(test_set, start=1):
        pred = predictor(user_id, item_id)
        err = pred - actual
        mse += err * err
        count += 1

        progress = (index / total) * 100
        if progress >= next_progress or index == total:
            elapsed = time.perf_counter() - started_at
            print(f"{label}: {progress:.0f}% ({index}/{total}) - {elapsed:.1f}s")
            next_progress += progress_step

    return math.sqrt(mse / count) if count else 0.0


def evaluate_precision_recall_at_k(test_set, ranker, k, label="Precision/Recall@K", progress_step=5):
    hits = 0
    total = 0
    total_cases = len(test_set)

    if total_cases == 0:
        print(f"{label}: 100% (no test data)")
        return 0.0, 0.0

    next_progress = progress_step
    started_at = time.perf_counter()

    for index, (user_id, item_id, _) in enumerate(test_set, start=1):
        ranked = ranker(user_id, k)
        total += 1
        if item_id in ranked:
            hits += 1

        progress = (index / total_cases) * 100
        if progress >= next_progress or index == total_cases:
            elapsed = time.perf_counter() - started_at
            print(f"{label}: {progress:.0f}% ({index}/{total_cases}) - {elapsed:.1f}s")
            next_progress += progress_step

    precision = (hits / total) / k if total else 0.0
    recall = hits / total if total else 0.0
    return precision, recall

## Execução 

In [7]:
data_dir = os.path.join(os.getcwd(), "data")
if not os.path.exists(data_dir):
    data_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))

review_path = os.path.join(data_dir, "review_final.json")
business_path = os.path.join(data_dir, "business_final.json")

print("Loading reviews and business categories...")
user_ratings = load_reviews(review_path)
categories = load_business_categories(business_path)
print(f"Loaded {len(user_ratings)} users and {len(categories)} businesses.")

train_user_ratings, test_set = split_train_test(user_ratings, min_ratings=2, test_ratio=0.2, seed=42)
if not test_set:
    train_user_ratings, test_set = split_train_test(user_ratings, min_ratings=1, test_ratio=0.2, seed=42)

item_counts, item_mean, global_mean = build_item_stats(train_user_ratings)

popularity_candidates = [item for item, _ in item_counts.most_common()]
all_items = list(item_mean.keys())

print(f"Train users: {len(train_user_ratings)} | Test cases: {len(test_set)} | Candidate items: {len(all_items)}")

K = 10

baseline_rmse = evaluate_rmse(
    test_set,
    lambda user_id, item_id: predict_baseline_rating(item_id, item_mean, global_mean),
    label="Baseline RMSE",
)
content_rmse = evaluate_rmse(
    test_set,
    lambda user_id, item_id: predict_content_rating(
        user_id,
        item_id,
        train_user_ratings,
        categories,
        global_mean,
    ),
    label="Content RMSE",
)

baseline_precision, baseline_recall = evaluate_precision_recall_at_k(
    test_set,
    lambda user_id, k: rank_baseline(user_id, train_user_ratings, popularity_candidates, item_mean, k),
    K,
    label=f"Baseline Precision/Recall@{K}",
)
content_precision, content_recall = evaluate_precision_recall_at_k(
    test_set,
    lambda user_id, k: rank_content(user_id, train_user_ratings, categories, all_items, k),
    K,
    label=f"Content Precision/Recall@{K}",
)

results_df = pd.DataFrame(
    [
        {
            "model": "baseline",
            "rmse": baseline_rmse,
            f"precision@{K}": baseline_precision,
            f"recall@{K}": baseline_recall,
        },
        {
            "model": "content",
            "rmse": content_rmse,
            f"precision@{K}": content_precision,
            f"recall@{K}": content_recall,
        },
    ]
)

results_df

Loading reviews and business categories...
Loaded 300 users and 18806 businesses.
Train users: 300 | Test cases: 5213 | Candidate items: 15913
Baseline RMSE: 10% (522/5213) - 0.0s
Baseline RMSE: 20% (1043/5213) - 0.0s
Baseline RMSE: 30% (1564/5213) - 0.0s
Baseline RMSE: 40% (2086/5213) - 0.0s
Baseline RMSE: 50% (2607/5213) - 0.0s
Baseline RMSE: 60% (3128/5213) - 0.0s
Baseline RMSE: 70% (3650/5213) - 0.0s
Baseline RMSE: 80% (4171/5213) - 0.0s
Baseline RMSE: 90% (4692/5213) - 0.0s
Baseline RMSE: 100% (5213/5213) - 0.0s
Content RMSE: 10% (522/5213) - 0.1s
Content RMSE: 20% (1043/5213) - 0.2s
Content RMSE: 30% (1564/5213) - 0.3s
Content RMSE: 40% (2086/5213) - 0.4s
Content RMSE: 50% (2607/5213) - 0.5s
Content RMSE: 60% (3128/5213) - 0.5s
Content RMSE: 70% (3650/5213) - 0.8s
Content RMSE: 80% (4171/5213) - 1.1s
Content RMSE: 90% (4692/5213) - 1.6s
Content RMSE: 100% (5213/5213) - 1.7s
Baseline Precision/Recall@10: 5% (261/5213) - 0.2s
Baseline Precision/Recall@10: 10% (522/5213) - 0.3s
Base

,model,rmse,precision@10,recall@10
0,baseline,1.142175,0.000422,0.004220
1,content,1.028866,0.000230,0.002302
